In [5]:
import pandas as pd
import numpy as np
from scipy.stats import false_discovery_control

# Input and output file names
input_file = "Raw-p-val.xlsx"
output_file = "Raw-p-val_FDR_corrected.xlsx"

# Sheet names to process (each represents a family of tests)
target_sheets = [
    "degree_centrality",
    "degree_centrality2",
    "betweenness_centrality",
    "betweenness_centrality2",
]

# Read Excel file
xls = pd.ExcelFile(input_file)

processed_sheets = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(input_file, sheet_name=sheet).copy()

    if sheet in target_sheets:
        if "p-value" not in df.columns:
            print(f"[SKIP] Sheet '{sheet}' does not contain a 'p-value' column.")
            processed_sheets[sheet] = df
            continue

        df["p-value"] = pd.to_numeric(df["p-value"], errors="coerce")
        mask = df["p-value"].notna()

        # Initialize columns
        df["FDR_BH_pvalue"] = np.nan
        df["FDR_BH_significant_0.05"] = np.nan

        if mask.sum() == 0:
            print(f"[SKIP] Sheet '{sheet}' has no valid p-values.")
        else:
            corrected_pvals = false_discovery_control(
                df.loc[mask, "p-value"].to_numpy(),
                method="bh",
            )

            df.loc[mask, "FDR_BH_pvalue"] = corrected_pvals

            # Convert boolean → integer explicitly
            df.loc[mask, "FDR_BH_significant_0.05"] = (
                corrected_pvals < 0.05
            ).astype(int)

            print(f"[DONE] Sheet '{sheet}' corrected for {mask.sum()} p-values.")

        # Enforce integer dtype where possible in "decision" column
        df["FDR_BH_significant_0.05"] = df["FDR_BH_significant_0.05"].astype("Int64")

    processed_sheets[sheet] = df

# ==========================================
# Save all sheets to a new Excel file
# ==========================================
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet, df in processed_sheets.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

print(f"\nFinished. New file saved as: {output_file}")

[DONE] Sheet 'degree_centrality' corrected for 32 p-values.
[DONE] Sheet 'degree_centrality2' corrected for 4 p-values.
[DONE] Sheet 'betweenness_centrality' corrected for 22 p-values.
[DONE] Sheet 'betweenness_centrality2' corrected for 2 p-values.

Finished. New file saved as: Raw-p-val_FDR_corrected.xlsx
